# Phase 1 : préparation des données

Données NASA POWER (MERRA-2), lac Monoun, 1983–2024. Objectif : produire les transactions binaires à charger dans Weka (FP-Growth).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.preparation import (
    THRESHOLDS, load_power_csv, check_missing, add_dtr, build_transactions,
    item_frequencies, to_binary, export_arff, export_arff_nominal, save_index,
)

RAW = ROOT / 'data' / 'raw' / 'LAC_MONOUN_POWER_Daily_1983-2024.csv'
PROCESSED = ROOT / 'data' / 'processed'

## 1. Chargement et contrôle des valeurs manquantes

NASA POWER code les valeurs manquantes par `-999`.

In [2]:
df = load_power_csv(RAW)
df, missing = check_missing(df)
print(df.shape)
missing

(15341, 7)


YEAR           0
DOY            0
PRECTOTCORR    0
T2M_MIN        0
T2M_MAX        0
WS2M           0
PS             0
dtype: int64

## 2. Statistiques descriptives

In [3]:
df.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
YEAR,15341.0,2003.50,12.12,1983.00,1993.00,2004.00,2014.00,2024.00
DOY,15341.0,183.13,105.45,1.00,92.00,183.00,274.00,366.00
PRECTOTCORR,15341.0,5.33,7.36,0.00,0.19,3.11,7.42,121.91
T2M_MIN,15341.0,17.08,1.55,9.58,16.35,17.32,18.10,21.89
T2M_MAX,15341.0,25.28,2.00,19.14,23.94,25.20,26.55,33.83
WS2M,15341.0,1.36,0.39,0.34,1.07,1.31,1.60,3.68
PS,15341.0,88.61,0.12,88.17,88.53,88.62,88.70,89.02


## 3. Amplitude thermique journalière

DTR = T2M_MAX − T2M_MIN, arrondie à 2 décimales comme les données sources.

In [4]:
df = add_dtr(df)
df['DTR'].describe().round(2)

count    15341.00
mean         8.20
std          2.64
min          1.77
25%          6.28
50%          7.66
75%          9.92
max         17.58
Name: DTR, dtype: float64

## 4. Discrétisation

Seuils de l'énoncé, utilisés tels quels.

**Règle des bornes :** une valeur égale à un seuil va dans la classe intermédiaire. L'énoncé définit les classes extrêmes par des inégalités strictes (< 1 mm, > 14 °C, < 88,4 kPa…), une valeur égale au seuil ne peut donc pas y appartenir. Pour la pluie, cela rejoint la convention ETCCDI (jour pluvieux : ≥ 1 mm). Seule divergence : l'ETCCDI classe 20 mm en « très fortes pluies » (R20mm : ≥ 20 mm), alors que l'énoncé (> 20 mm) le met en pluie modérée ; on suit l'énoncé (2 jours concernés).

**Saisons :** sèche (DOY 320–060), transition (061–150), pic de mousson (151–319). Les années bissextiles (décalage d'un jour après le 29 février) sont négligées.

In [5]:
transactions = build_transactions(df, THRESHOLDS)
transactions.head()

,Season,Rain,Tmin,DTR,Wind,Pressure
0,Season_Dry,Rain_None,Tmin_Cold,DTR_Moderate,Wind_Moderate,Pressure_Normal
1,Season_Dry,Rain_None,Tmin_Cold,DTR_Moderate,Wind_Moderate,Pressure_Normal
2,Season_Dry,Rain_None,Tmin_Cold,DTR_Moderate,Wind_Moderate,Pressure_Normal
3,Season_Dry,Rain_None,Tmin_Cold,DTR_Moderate,Wind_Moderate,Pressure_High_Stable
4,Season_Dry,Rain_None,Tmin_Cold,DTR_Moderate,Wind_Moderate,Pressure_High_Stable


### Répartition des items

In [6]:
item_frequencies(transactions)

,variable,item,count,share
0,DTR,DTR_Suppressed,8566,0.5584
1,DTR,DTR_Moderate,6441,0.4199
2,DTR,DTR_Extreme,334,0.0218
3,Pressure,Pressure_Normal,11167,0.7279
4,Pressure,Pressure_High_Stable,3544,0.2310
5,Pressure,Pressure_Critical_Drop,630,0.0411
6,Rain,Rain_Moderate,9503,0.6195
7,Rain,Rain_None,5201,0.3390
8,Rain,Rain_Extreme_Downpour,637,0.0415
9,Season,Season_Monsoon_Peak,7098,0.4627


## 5. Encodage binaire

18 colonnes 0/1, une par item. Contrôle : chaque jour contient exactement 6 items.

In [7]:
binary = to_binary(transactions, THRESHOLDS)
print(binary.shape)
binary.sum().sort_values()

(15341, 18)


Wind_High_Shear               5
Tmin_Severe_Cooling          16
DTR_Extreme                 334
Pressure_Critical_Drop      630
Rain_Extreme_Downpour       637
Tmin_Cold                   767
Pressure_High_Stable       3544
Season_Transition          3780
Season_Dry                 4463
Wind_Moderate              5158
Rain_None                  5201
DTR_Moderate               6441
Season_Monsoon_Peak        7098
DTR_Suppressed             8566
Rain_Moderate              9503
Wind_Calm                 10178
Pressure_Normal           11167
Tmin_Warm                 14558
dtype: int64

## 6. Export

- `transactions.csv` : items par jour, avec YEAR et DOY (pour la phase 3)
- `transactions_binary.arff` : entrée de Weka FP-Growth (`positiveIndex = 2`, la valeur `1` signifie « item présent »)
- `transactions_nominal.arff` : entrée de Weka Apriori (6 attributs nominaux, chaque valeur est un item)
- `index.csv` : correspondance ligne → (YEAR, DOY)

In [8]:
df[['YEAR', 'DOY']].join(transactions).to_csv(PROCESSED / 'transactions.csv', index=False)
export_arff(binary, PROCESSED / 'transactions_binary.arff')
export_arff_nominal(transactions, PROCESSED / 'transactions_nominal.arff')
save_index(df, PROCESSED / 'index.csv')
sorted(p.name for p in PROCESSED.iterdir() if p.is_file())

['.gitkeep',
 'index.csv',
 'transactions.csv',
 'transactions_binary.arff',
 'transactions_nominal.arff']